# 🤖 Notebook 03 — Model Training & Evaluation

Trains all three classifiers, compares them, and deep-dives into the best model.

**Contents**
1. Setup & data loading
2. Train all models
3. Metrics comparison table
4. ROC curve comparison
5. Confusion matrices
6. Feature importance (best model)
7. Threshold tuning
8. Save & load best model
9. Predict on new customers

In [ ]:
import sys, os, warnings
sys.path.insert(0, '../src')
sys.path.insert(0, '../visualization')
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

from data_preprocessing import full_preprocessing_pipeline
from train_model import build_models, save_model, load_model
from evaluate_model import evaluate_model, plot_confusion_matrix, plot_roc_curves, plot_feature_importance

RAW = '../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv'
MODELS_DIR = '../models'

## 1. Load Preprocessed Data

In [ ]:
X_train, X_test, y_train, y_test, scaler, feature_names = full_preprocessing_pipeline(RAW, processed_dir='../data/processed')
print(f'Training features: {len(feature_names)}')
print(f'Train/Test split:  {len(X_train):,} / {len(X_test):,}')

## 2. Train All Models

In [ ]:
models = build_models()
results = {}

for name, model in models.items():
    print(f'Training {name}...')
    model.fit(X_train, y_train)
    metrics, y_pred, y_prob = evaluate_model(model, X_test, y_test, model_name=name)
    results[name] = {'model': model, 'metrics': metrics, 'y_pred': y_pred, 'y_prob': y_prob, 'y_test': y_test}

## 3. Metrics Comparison Table

In [ ]:
rows = []
for name, res in results.items():
    m = res['metrics']
    rows.append({'Model': name, 'Accuracy': m['accuracy'], 'Precision': m['precision'],
                 'Recall': m['recall'], 'F1': m['f1'], 'ROC-AUC': m['roc_auc']})

comparison = pd.DataFrame(rows).set_index('Model').round(4)
comparison['Best ROC-AUC'] = comparison['ROC-AUC'] == comparison['ROC-AUC'].max()
print(comparison.to_string())
comparison.drop(columns=['Best ROC-AUC'])

## 4. ROC Curve Comparison

In [ ]:
from sklearn.metrics import roc_curve

fig, ax = plt.subplots(figsize=(8,6))
ax.plot([0,1],[0,1],'--',color='gray',label='Random (AUC=0.50)')
colors = ['#2563eb','#dc2626','#16a34a']
for (name, res), color in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    ax.plot(fpr, tpr, color=color, linewidth=2, label=f"{name} (AUC={res['metrics']['roc_auc']:.4f})")
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — All Models', fontweight='bold')
ax.legend()
ax.grid(linestyle=':')
plt.tight_layout()
plt.show()

## 5. Confusion Matrices

In [ ]:
from sklearn.metrics import confusion_matrix

fig, axes = plt.subplots(1, len(results), figsize=(5*len(results), 4))
if len(results) == 1: axes = [axes]

for ax, (name, res) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, res['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['No Churn','Churn'], yticklabels=['No Churn','Churn'],
                cbar=False, annot_kws={'size':14,'weight':'bold'})
    ax.set_title(name, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.suptitle('Confusion Matrices', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 6. Feature Importance (Best Model)

In [ ]:
best_name = max(results, key=lambda n: results[n]['metrics']['roc_auc'])
best_model = results[best_name]['model']
print(f'Best model: {best_name}')

if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
elif hasattr(best_model, 'coef_'):
    importances = np.abs(best_model.coef_[0])

indices = np.argsort(importances)[::-1][:20]
top_features = [feature_names[i] for i in indices]
top_values   = importances[indices]

fig, ax = plt.subplots(figsize=(10,7))
colors = plt.cm.Blues(np.linspace(0.4, 0.9, 20))[::-1]
ax.barh(range(20), top_values[::-1], color=colors[::-1], edgecolor='white')
ax.set_yticks(range(20))
ax.set_yticklabels(top_features[::-1])
ax.set_xlabel('Feature Importance')
ax.set_title(f'{best_name} — Top 20 Feature Importances', fontweight='bold')
ax.grid(axis='x', linestyle=':')
plt.tight_layout()
plt.show()

## 7. Threshold Tuning

The default threshold of 0.5 may not be optimal. Higher recall catches more churners; higher precision reduces false alerts.

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

y_prob_best = results[best_name]['y_prob']
thresholds = np.arange(0.2, 0.8, 0.05)

rows = []
for t in thresholds:
    y_pred_t = (y_prob_best >= t).astype(int)
    rows.append({
        'Threshold': round(t, 2),
        'Precision': precision_score(y_test, y_pred_t, zero_division=0),
        'Recall':    recall_score(y_test, y_pred_t, zero_division=0),
        'F1':        f1_score(y_test, y_pred_t, zero_division=0),
    })

thresh_df = pd.DataFrame(rows).set_index('Threshold').round(4)

fig, ax = plt.subplots(figsize=(10,4))
thresh_df.plot(ax=ax, marker='o')
ax.axvline(0.5, color='black', linestyle='--', alpha=0.5, label='Default (0.5)')
ax.set_xlabel('Threshold')
ax.set_title('Precision / Recall / F1 vs Classification Threshold', fontweight='bold')
ax.legend()
ax.grid(linestyle=':')
plt.tight_layout()
plt.show()

thresh_df

## 8. Save & Reload Best Model

In [ ]:
from train_model import save_model, load_model

path = save_model(best_model, scaler, feature_names, path=f'{MODELS_DIR}/churn_model.pkl')

# Reload and verify
bundle = load_model(path)
print(f'Bundle keys: {list(bundle.keys())}')
print(f'Model type:  {type(bundle["model"]).__name__}')
print(f'Features:    {len(bundle["feature_names"])} columns')

## 9. Predict on New Customers

Demonstrate the full inference path on 3 fictional customer profiles.

In [ ]:
new_customers = pd.DataFrame([
    {
        'customerID': 'RISK-001',
        'gender': 'Female', 'SeniorCitizen': 0, 'Partner': 'No', 'Dependents': 'No',
        'tenure': 2, 'PhoneService': 'Yes', 'MultipleLines': 'No',
        'InternetService': 'Fiber optic', 'OnlineSecurity': 'No', 'OnlineBackup': 'No',
        'DeviceProtection': 'No', 'TechSupport': 'No', 'StreamingTV': 'Yes',
        'StreamingMovies': 'Yes', 'Contract': 'Month-to-month',
        'PaperlessBilling': 'Yes', 'PaymentMethod': 'Electronic check',
        'MonthlyCharges': 95.5, 'TotalCharges': '191.0'
    },
    {
        'customerID': 'SAFE-002',
        'gender': 'Male', 'SeniorCitizen': 0, 'Partner': 'Yes', 'Dependents': 'Yes',
        'tenure': 58, 'PhoneService': 'Yes', 'MultipleLines': 'Yes',
        'InternetService': 'DSL', 'OnlineSecurity': 'Yes', 'OnlineBackup': 'Yes',
        'DeviceProtection': 'Yes', 'TechSupport': 'Yes', 'StreamingTV': 'No',
        'StreamingMovies': 'No', 'Contract': 'Two year',
        'PaperlessBilling': 'No', 'PaymentMethod': 'Bank transfer (automatic)',
        'MonthlyCharges': 64.0, 'TotalCharges': '3712.0'
    },
    {
        'customerID': 'MED-003',
        'gender': 'Male', 'SeniorCitizen': 1, 'Partner': 'No', 'Dependents': 'No',
        'tenure': 18, 'PhoneService': 'Yes', 'MultipleLines': 'No',
        'InternetService': 'Fiber optic', 'OnlineSecurity': 'No', 'OnlineBackup': 'Yes',
        'DeviceProtection': 'No', 'TechSupport': 'No', 'StreamingTV': 'No',
        'StreamingMovies': 'No', 'Contract': 'One year',
        'PaperlessBilling': 'Yes', 'PaymentMethod': 'Mailed check',
        'MonthlyCharges': 73.5, 'TotalCharges': '1323.0'
    },
])

from train_model import predict_churn
preds = predict_churn(new_customers.drop(columns=['customerID']), bundle)
preds.insert(0, 'customerID', new_customers['customerID'].values)
preds[['customerID','tenure','Contract','MonthlyCharges','ChurnProbability','ChurnPrediction']]